# 05 — Modelado binario con XGBoost

**Proyecto:** VIGÍA (Sentry) — TFC
**Objetivo:** entrenar un detector binario (BENIGN vs ATAQUE) y medir el impacto
de `Destination_Port` mediante dos experimentos comparables.

**Entradas:**
- `cic-ids2017-splits/train.parquet`
- `cic-ids2017-splits/val.parquet`
- `cic-ids2017-splits/test.parquet`

**Decisiones que se toman:**
- Métrica principal: **recall en ATAQUE** (porque un falso negativo es lo peor en seguridad).
- Dos modelos: A (con `Destination_Port`) y B (sin `Destination_Port`).
- `scale_pos_weight` para compensar el desbalance 4:1.
- Semilla fija (42) para reproducibilidad.

**Lo que NO se toca en este notebook:**
- `test.parquet` no se usa hasta la evaluación final.
- Hiperparámetros no se afinan todavía (baseline primero).

In [ ]:
# ============================================
# Reconstrucción completa con muestreo de train
# ============================================
import os
from google.colab import drive

GITHUB_USER = "AguCS231"
REPO_NAME   = "sentry"
REPO_LOCAL  = f"/content/{REPO_NAME}"
BASE_SPLITS = "/content/drive/MyDrive/cic-ids2017-splits"

if not os.path.exists(REPO_LOCAL):
    %cd /content
    !git clone https://github.com/{GITHUB_USER}/{REPO_NAME}.git
    %cd {REPO_LOCAL}
    !git config --global user.email "srcasares2@gmail.com"
    !git config --global user.name "{GITHUB_USER}"
else:
    %cd {REPO_LOCAL}

drive.mount('/content/drive')

from pyspark.sql import SparkSession
from pyspark.sql.types import NumericType

spark = SparkSession.builder \
    .appName("VIGIA_Modelado_Binario") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print(f"✅ Spark {spark.version}\n")

train_df = spark.read.parquet(f"{BASE_SPLITS}/train.parquet")
val_df   = spark.read.parquet(f"{BASE_SPLITS}/val.parquet")
test_df  = spark.read.parquet(f"{BASE_SPLITS}/test.parquet")

# Definir features
excluir = {"Label", "target_bin", "target_family"}
feature_cols_A = [
    c for c in train_df.columns
    if c not in excluir and isinstance(train_df.schema[c].dataType, NumericType)
]
feature_cols_B = [c for c in feature_cols_A if c != "Destination_Port"]

# ============================================
# Muestreo estratificado del 50% en train
# ============================================
print("Muestreando train al 50% (estratificado)...")
train_benign = train_df.filter("target_bin = 0").sample(fraction=0.5, seed=42)
train_attack = train_df.filter("target_bin = 1").sample(fraction=0.5, seed=42)
train_sample = train_benign.union(train_attack)

# Cachear la muestra
train_sample.cache()

n_full = train_df.count()
n_sample = train_sample.count()
print(f"\nTrain completo:   {n_full:>10,}")
print(f"Train muestreado: {n_sample:>10,} ({n_sample/n_full*100:.1f}%)")
print(f"\nVal:  {val_df.count():>10,}")
print(f"Test: {test_df.count():>10,}")
print(f"\nFeatures A (con puerto): {len(feature_cols_A)}")
print(f"Features B (sin puerto): {len(feature_cols_B)}")

## 1. Definición de features

Dos conjuntos:
- **A (con puerto):** todas las features numéricas, incluida `Destination_Port`.
- **B (sin puerto):** lo mismo, excluyendo `Destination_Port`.

**Motivo de seguridad:** en el EDA vimos que cinco clases tienen puertos
exclusivos (Bot→8080, FTP-Patator→21, SSH-Patator→22, Heartbleed→444,
Infiltration→444). Si el modelo A aprende "puerto 21 → ataque", en producción
basta con que el atacante cambie de puerto para evadir la detección.

In [ ]:
# ============================================
# Definir dos conjuntos de features
# ============================================
from pyspark.sql.types import NumericType

# Excluir del modelo: Label (texto), targets
excluir = {"Label", "target_bin", "target_family"}

# Todas las numéricas menos las excluidas
feature_cols_A = [
    c for c in train_df.columns
    if c not in excluir and isinstance(train_df.schema[c].dataType, NumericType)
]

# Sin Destination_Port
feature_cols_B = [c for c in feature_cols_A if c != "Destination_Port"]

print(f"Features A (con puerto): {len(feature_cols_A)}")
print(f"Features B (sin puerto): {len(feature_cols_B)}")
print(f"\nÚnica diferencia: 'Destination_Port'")
print(f"¿Está en A?: {'Destination_Port' in feature_cols_A}")
print(f"¿Está en B?: {'Destination_Port' in feature_cols_B}")

## 2. Conversión a numpy

XGBoost trabaja con arrays de numpy, no con DataFrames de Spark. Convertimos:

- `train_df` → `X_train_A`, `X_train_B`, `y_train`
- `val_df`   → `X_val_A`,   `X_val_B`,   `y_val`

**Decisión técnica:** usamos `float32` en lugar de `float64` para reducir el uso
de memoria a la mitad (2M filas × 78 features × 8 bytes = 1.25 GB en float64,
solo 620 MB en float32). La pérdida de precisión es irrelevante para XGBoost.

**Por qué no usamos el DataFrame de Spark directamente:** XGBoost es una
librería single-node. Repartir el entrenamiento en Spark no aporta y complica
la reproducibilidad.

In [ ]:
# ============================================
# Conversión Spark → numpy (float32, train muestreado)
# ============================================
import numpy as np
import time, gc

print("Convirtiendo train muestreado...")
t0 = time.time()
train_pd = train_sample.select(feature_cols_A + ["target_bin"]).toPandas()
print(f"✅ Convertido en {time.time()-t0:.1f}s | shape: {train_pd.shape}")

X_train_A = train_pd[feature_cols_A].values.astype(np.float32)
y_train   = train_pd["target_bin"].values.astype(np.int8)

del train_pd
gc.collect()

print(f"\nX_train_A: {X_train_A.shape}  ({X_train_A.nbytes/1e6:.0f} MB)")
print(f"y_train:   {y_train.shape}")
print(f"\nDistribución train:")
print(f"  BENIGN: {(y_train==0).sum():,}")
print(f"  ATAQUE: {(y_train==1).sum():,} ({(y_train==1).mean()*100:.2f}%)")

In [ ]:
# ============================================
# Conversión de validation
# ============================================
import gc

print("Convirtiendo val...")
val_pd = val_df.select(feature_cols_A + ["target_bin"]).toPandas()

X_val_A = val_pd[feature_cols_A].values.astype(np.float32)
X_val_B = val_pd[feature_cols_B].values.astype(np.float32)
y_val   = val_pd["target_bin"].values.astype(np.int8)

del val_pd
gc.collect()

print(f"✅ X_val_A: {X_val_A.shape} | X_val_B: {X_val_B.shape} | y_val: {y_val.shape}")

print(f"\nTrain → BENIGN: {(y_train==0).sum():,} | ATAQUE: {(y_train==1).sum():,} "
      f"({(y_train==1).mean()*100:.2f}%)")
print(f"Val   → BENIGN: {(y_val==0).sum():,} | ATAQUE: {(y_val==1).sum():,} "
      f"({(y_val==1).mean()*100:.2f}%)")

In [ ]:
# ============================================
# Entrenar modelo A (con Destination_Port)
# ============================================
import xgboost as xgb
import time

# Calcular scale_pos_weight
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
spw = neg / pos
print(f"scale_pos_weight: {spw:.2f}  (neg={neg:,} / pos={pos:,})\n")

modelo_A = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    max_depth=6,
    n_estimators=100,
    learning_rate=0.1,
    scale_pos_weight=spw,
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
    verbosity=0,
)

print("Entrenando modelo A (2-5 min)...")
t0 = time.time()
modelo_A.fit(
    X_train_A, y_train,
    eval_set=[(X_val_A, y_val)],
    verbose=False,
)
print(f"✅ Entrenado en {time.time()-t0:.1f}s")

## 4. Evaluación del modelo A

**Métrica principal: recall en ATAQUE.** En ciberseguridad, un falso negativo
(ataque clasificado como BENIGN) es mucho peor que un falso positivo (BENIGN
clasificado como ataque). El recall mide exactamente eso: de todos los ataques
reales, ¿cuántos detectamos?

**Métricas secundarias:**
- Precision: de lo que marcamos como ataque, ¿cuánto lo era realmente?
- F1: media armónica de precision y recall.
- AUC-ROC: capacidad de separar las dos clases con cualquier umbral.

**Motivo de seguridad:** un IDS con recall bajo es un IDS que deja pasar
ataques. Un IDS con precision baja genera ruido. En un SOC, el ruido se tolera
hasta cierto punto; dejar pasar ataques, no.

In [ ]:
# ============================================
# Evaluación del modelo A en validation
# ============================================
from sklearn.metrics import (
    recall_score, precision_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)
import numpy as np

# Predicciones
y_pred_A = modelo_A.predict(X_val_A)
y_proba_A = modelo_A.predict_proba(X_val_A)[:, 1]

# Métricas
print("=== Modelo A (con Destination_Port) — Validation ===\n")
print(f"Recall (ATAQUE):    {recall_score(y_val, y_pred_A):.4f}")
print(f"Precision (ATAQUE): {precision_score(y_val, y_pred_A):.4f}")
print(f"F1 (ATAQUE):        {f1_score(y_val, y_pred_A):.4f}")
print(f"AUC-ROC:            {roc_auc_score(y_val, y_proba_A):.4f}")

print("\n=== Matriz de confusión ===")
cm = confusion_matrix(y_val, y_pred_A)
print(f"                 Pred BENIGN  Pred ATAQUE")
print(f"Real BENIGN      {cm[0,0]:>10,}  {cm[0,1]:>11,}")
print(f"Real ATAQUE      {cm[1,0]:>10,}  {cm[1,1]:>11,}")

tn, fp, fn, tp = cm.ravel()
print(f"\nInterpretación:")
print(f"  Ataques detectados:   {tp:,} / {tp+fn:,}  ({tp/(tp+fn)*100:.2f}%)")
print(f"  Ataques NO detectados: {fn:,}  ← falsos negativos (crítico)")
print(f"  Falsas alarmas:        {fp:,}  ← falsos positivos")

## 5. Entrenamiento del modelo B (sin Destination_Port)

Mismo XGBoost, mismos hiperparámetros, mismos datos. **Única diferencia:**
excluye `Destination_Port`.

**Objetivo:** medir cuánto del rendimiento del modelo A dependía de esa
feature. Si A y B dan resultados parecidos, el puerto no era crítico. Si A es
mucho mejor, el modelo estaba aprendiendo un atajo.

**Motivo de seguridad:** si el puerto es la clave del rendimiento, un atacante
que cambie de puerto evade la detección. El modelo B es el detector honesto.

In [ ]:
# ============================================
# Convertir train sin Destination_Port y entrenar modelo B
# ============================================
import gc, time

# Liberar memoria del modelo A antes de crear otro dataset
del X_train_A
gc.collect()

print("Convirtiendo train sin Destination_Port...")
t0 = time.time()
train_pd_B = train_sample.select(feature_cols_B + ["target_bin"]).toPandas()
X_train_B = train_pd_B[feature_cols_B].values.astype(np.float32)
del train_pd_B
gc.collect()
print(f"✅ X_train_B: {X_train_B.shape}  ({X_train_B.nbytes/1e6:.0f} MB) en {time.time()-t0:.1f}s")

# Entrenar modelo B
spw = (y_train == 0).sum() / (y_train == 1).sum()

modelo_B = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    max_depth=6,
    n_estimators=100,
    learning_rate=0.1,
    scale_pos_weight=spw,
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
    verbosity=0,
)

print("\nEntrenando modelo B...")
t0 = time.time()
modelo_B.fit(
    X_train_B, y_train,
    eval_set=[(X_val_B, y_val)],
    verbose=False,
)
print(f"✅ Entrenado en {time.time()-t0:.1f}s")

In [ ]:
# ============================================
# Evaluación del modelo B en validation
# ============================================
y_pred_B = modelo_B.predict(X_val_B)
y_proba_B = modelo_B.predict_proba(X_val_B)[:, 1]

print("=== Modelo B (sin Destination_Port) — Validation ===\n")
print(f"Recall (ATAQUE):    {recall_score(y_val, y_pred_B):.4f}")
print(f"Precision (ATAQUE): {precision_score(y_val, y_pred_B):.4f}")
print(f"F1 (ATAQUE):        {f1_score(y_val, y_pred_B):.4f}")
print(f"AUC-ROC:            {roc_auc_score(y_val, y_proba_B):.4f}")

cm_B = confusion_matrix(y_val, y_pred_B)
tn, fp, fn, tp = cm_B.ravel()
print(f"\nAtaques detectados:   {tp:,} / {tp+fn:,}  ({tp/(tp+fn)*100:.2f}%)")
print(f"Ataques NO detectados: {fn:,}")
print(f"Falsas alarmas:        {fp:,}")

In [ ]:
# ============================================
# Comparativa modelo A vs modelo B
# ============================================
print("=== Comparativa A vs B (validation) ===\n")
print(f"{'Métrica':<20} {'Modelo A':>12} {'Modelo B':>12} {'Δ (A-B)':>12}")
print("-" * 60)

metricas = {
    "Recall ATAQUE":    (recall_score(y_val, y_pred_A),    recall_score(y_val, y_pred_B)),
    "Precision ATAQUE": (precision_score(y_val, y_pred_A), precision_score(y_val, y_pred_B)),
    "F1 ATAQUE":        (f1_score(y_val, y_pred_A),        f1_score(y_val, y_pred_B)),
    "AUC-ROC":          (roc_auc_score(y_val, y_proba_A),  roc_auc_score(y_val, y_proba_B)),
}

for nombre, (a, b) in metricas.items():
    delta = a - b
    print(f"{nombre:<20} {a:>12.4f} {b:>12.4f} {delta:>+12.4f}")

print("\nInterpretación:")
delta_recall = metricas["Recall ATAQUE"][0] - metricas["Recall ATAQUE"][1]
if abs(delta_recall) < 0.005:
    print("  ✅ Diferencia < 0.5% en recall → el puerto NO es crítico.")
    print("     Recomendación: usar modelo B (más robusto a evasión).")
elif delta_recall > 0:
    print(f"  ⚠️ Modelo A detecta {delta_recall*100:.2f}% más ataques que B.")
    print("     El puerto aporta señal → investigar qué clases dependen de él.")
else:
    print(f"  ⚠️ Modelo B es MEJOR que A. El puerto introducía ruido.")

In [ ]:
# ============================================
# Guardar modelos A y B
# ============================================
import os, json

MODELS_DIR = "/content/sentry/src/models"
os.makedirs(MODELS_DIR, exist_ok=True)

# Guardar con el formato nativo de XGBoost (más robusto que pickle)
modelo_A.save_model(f"{MODELS_DIR}/binario_A_con_puerto.json")
modelo_B.save_model(f"{MODELS_DIR}/binario_B_sin_puerto.json")

# Guardar metadatos: features usadas, métricas, semilla
meta = {
    "fecha": "2026-09-29",
    "features_A": feature_cols_A,
    "features_B": feature_cols_B,
    "n_train": int(len(y_train)),
    "n_val": int(len(y_val)),
    "muestreo_train": "50% estratificado (seed=42) por limitaciones de RAM",
    "hiperparametros": {
        "max_depth": 6, "n_estimators": 100, "learning_rate": 0.1,
        "scale_pos_weight": float(spw), "random_state": 42,
    },
    "metricas_val_A": {
        "recall": float(recall_score(y_val, y_pred_A)),
        "precision": float(precision_score(y_val, y_pred_A)),
        "f1": float(f1_score(y_val, y_pred_A)),
        "auc": float(roc_auc_score(y_val, y_proba_A)),
    },
    "metricas_val_B": {
        "recall": float(recall_score(y_val, y_pred_B)),
        "precision": float(precision_score(y_val, y_pred_B)),
        "f1": float(f1_score(y_val, y_pred_B)),
        "auc": float(roc_auc_score(y_val, y_proba_B)),
    },
}

with open(f"{MODELS_DIR}/binario_metadata.json", "w") as f:
    json.dump(meta, f, indent=2)

print("✅ Modelos guardados:")
!ls -lh {MODELS_DIR}/

In [ ]:
# ============================================
# Diario y commit
# ============================================
%cd /content/sentry

from datetime import datetime
fecha = datetime.now().strftime("%Y-%m-%d")

entrada = f"""

---

## {fecha} — Modelado binario (baseline)

**Qué probé:**
- XGBoost binario (BENIGN vs ATAQUE) con dos conjuntos de features:
  - Modelo A: 78 features (incluye Destination_Port).
  - Modelo B: 77 features (excluye Destination_Port).
- Muestreo estratificado del 50% del train (991.712 filas) por límites de RAM en Colab.
- scale_pos_weight = 4.07 (compensa desbalance BENIGN/ATAQUE).
- Validación sobre val.parquet completo (424.173 filas).

**Resultados en validation:**
| Métrica          | Modelo A | Modelo B | Δ (A-B) |
|------------------|----------|----------|---------|
| Recall ATAQUE    | 0.9993   | 0.9984   | +0.0009 |
| Precision ATAQUE | 0.9949   | 0.9917   | +0.0032 |
| F1 ATAQUE        | 0.9971   | 0.9950   | +0.0021 |
| AUC-ROC          | 0.9999   | 0.9999   | +0.0001 |

**Interpretación:**
- El puerto NO es un atajo crítico en el modelo binario.
- Diferencia de recall: 0.09% (77 ataques de 83K en val).
- Métricas casi saturadas (AUC 0.9999) → el binario es demasiado fácil en este dataset.

**Decisión:**
- Se adopta el modelo B (sin Destination_Port) como baseline del proyecto por ser
  más robusto frente a evasión por cambio de puerto.
- El experimento con/sin puerto se repetirá en el modelado por familias, donde
  las clases FTP-Patator y SSH-Patator son independientes y el puerto puede ser
  más relevante.

**Limitaciones conocidas:**
- Métricas saturadas en binario → no permiten diferenciar bien los dos modelos.
- Train al 50% por restricciones de memoria (no por decisión metodológica).
- Sin ajuste de hiperparámetros (baseline con valores por defecto razonables).

**Próximos pasos:**
- Evaluación final sobre test.parquet (una sola vez).
- Notebook 06: modelado por familias (Fase 2).
"""

with open("docs/diario.md", "a") as f:
    f.write(entrada)

print("✅ Diario actualizado")

!git add .
!git commit -m "Modelado binario: modelos A (con puerto) y B (sin puerto) + comparativa"

In [ ]:
from getpass import getpass
TOKEN = getpass("Token: ")
!git remote set-url origin https://{TOKEN}@github.com/AguCS231/sentry.git
!git push

## 6. Evaluación final del modelo B sobre test

Se evalúa **una sola vez** sobre `test.parquet`, que no se ha tocado hasta ahora.

**Motivo de seguridad:** evaluar varias veces sobre test equivale a usarlo como conjunto de validación. Se pierde la imparcialidad de la métrica y las cifras dejan de ser defendibles.

In [ ]:
# ============================================
# Evaluación final del modelo B en test
# ============================================
from sklearn.metrics import (
    recall_score, precision_score, f1_score,
    roc_auc_score, confusion_matrix
)

# Cargar test.parquet
test_df = spark.read.parquet(f"{BASE_SPLITS}/test.parquet")

# Convertir SOLO las features que usa el modelo B (sin puerto)
test_pd = test_df.select(feature_cols_B + ["target_bin"]).toPandas()

X_test_B = test_pd[feature_cols_B].values.astype(np.float32)
y_test   = test_pd["target_bin"].values.astype(np.int8)

del test_pd
import gc; gc.collect()

print(f"Test cargado: {X_test_B.shape}")

# Predicciones del modelo B (el que elegimos en la comparativa)
y_pred_test = modelo_B.predict(X_test_B)
y_proba_test = modelo_B.predict_proba(X_test_B)[:, 1]

# Métricas
print("\n=== Modelo B en TEST (evaluación final) ===\n")
print(f"Recall (ATAQUE):    {recall_score(y_test, y_pred_test):.4f}")
print(f"Precision (ATAQUE): {precision_score(y_test, y_pred_test):.4f}")
print(f"F1 (ATAQUE):        {f1_score(y_test, y_pred_test):.4f}")
print(f"AUC-ROC:            {roc_auc_score(y_test, y_proba_test):.4f}")

cm = confusion_matrix(y_test, y_pred_test)
tn, fp, fn, tp = cm.ravel()
print(f"\nAtaques detectados:   {tp:,} / {tp+fn:,}  ({tp/(tp+fn)*100:.2f}%)")
print(f"Ataques NO detectados: {fn:,}")
print(f"Falsas alarmas:        {fp:,}")

## 7. Análisis del rendimiento sobre la familia "Otros"

Recordemos que `Otros` (Infiltration + Heartbleed, 47 muestras) se reservó **íntegramente para test** y nunca se vio en entrenamiento.

**Este es el experimento de generalización:** ¿detecta el modelo ataques que no ha visto?

**Motivo de seguridad:** en producción, los ataques nuevos no están en el train. Un modelo que solo detecta lo que ha memorizado es trivialmente evadible.

In [ ]:
# ============================================
# Recall específico sobre la familia "Otros"
# ============================================
test_full = spark.read.parquet(f"{BASE_SPLITS}/test.parquet")
otros_df = test_full.filter("target_family = 'Otros'")

# Convertir solo esas filas
otros_pd = otros_df.select(feature_cols_B + ["Label", "target_bin"]).toPandas()
X_otros = otros_pd[feature_cols_B].values.astype(np.float32)

pred_otros = modelo_B.predict(X_otros)
otros_pd["prediccion"] = pred_otros

print("=== Comportamiento sobre 'Otros' (ataques NO vistos) ===\n")
print(otros_pd.groupby(["Label", "prediccion"]).size().to_string())

# Recall global de la familia Otros
recall_otros = (pred_otros == 1).mean()
print(f"\nRecall en 'Otros': {recall_otros*100:.2f}%  ({pred_otros.sum()} de {len(pred_otros)} detectados)")

In [ ]:
%cd /content/sentry

from datetime import datetime
fecha = datetime.now().strftime("%Y-%m-%d")

entrada = f"""

---

## {fecha} — Evaluación final del modelo binario sobre test

**Qué probé:**
- Evaluación única del modelo B (sin Destination_Port) sobre test.parquet.
- Cálculo del recall específico sobre la familia "Otros" (Infiltration + Heartbleed, 47 muestras no vistas en entrenamiento).

**Resultados en test:**
- (rellenar con las cifras de la celda 2)
- Recall familia "Otros": (rellenar)

**Interpretación:**
- (rellenar tras ver los resultados)

**Decisión:**
- Modelo B queda cerrado como baseline del proyecto. No se reajusta.
- El notebook 06 arranca el modelado por familias (Fase 2).

**Próximos pasos:**
- Modelo multi-clase por familias (7 clases).
"""

with open("docs/diario.md", "a") as f:
    f.write(entrada)

print("✅ Diario actualizado")

!git add .
!git commit -m "Evaluación final del modelo binario en test + análisis de generalización"
!git push

In [ ]:
from getpass import getpass
TOKEN = getpass("Token: ")
!git remote set-url origin https://{TOKEN}@github.com/AguCS231/sentry.git
!git push